## Task 2: Sentiment Analysis and Topic Modeling Pipeline

In [ ]:
import os
import pandas as pd
from transformers import pipeline


In [ ]:
print("🧠 Loading Hugging Face Sentiment Analysis model (DistilBERT)...")
# Initialize a lightweight, highly accurate deep learning transformer model
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

In [ ]:
def assign_business_theme(text):
    """Categorizes reviews into specific operational themes using keyword matching."""
    text_lower = str(text).lower()
    
    if any(word in text_lower for word in ['login', 'otp', 'password', 'sign in', 'register', 'account']):
        return "Account Access & Authentication"
    elif any(word in text_lower for word in ['transfer', 'send', 'money', 'payment', 'receive', 'transaction', 'cbe birr', 'amole']):
        return "Transactions & Fund Transfers"
    elif any(word in text_lower for word in ['slow', 'crash', 'freeze', 'stuck', 'loading', 'network', 'error', 'bug']):
        return "App Performance & Stability"
    elif any(word in text_lower for word in ['ui', 'interface', 'beautiful', 'clean', 'update', 'design', 'look']):
        return "UI & Design Feedback"
    else:
        return "General User Feedback"

In [ ]:
# 1. Load the cleaned dataset from Task 1
input_path = '../data/raw/cleaned_reviews.csv'
if not os.path.exists(input_path):
    # Backup path check in case notebook is running outside the notebooks directory
    input_path = 'data/raw/cleaned_reviews.csv'

print(f"📖 Reading dataset from {input_path}...")
df = pd.read_csv(input_path)

In [ ]:
# 2. Run Sentiment Analysis via AI Transformers
print("⚡ Analyzing text sentiments (this may take a moment)...")
reviews_list = df['review'].astype(str).tolist()

# Batch process to save computing memory
predictions = sentiment_analyzer(reviews_list, truncation=True)

# Extract simple labels and confidence weights
df['sentiment'] = [pred['label'] for pred in predictions]
df['sentiment_score'] = [round(pred['score'], 4) for pred in predictions]

In [ ]:
# 3. Apply Custom Rule-Based Topic Modeling
print("🏷️ Grouping text reviews into operational business themes...")
df['theme'] = df['review'].apply(assign_business_theme)

In [ ]:
# 4. Save the Enriched Dataset to data/processed/
output_dir = '../data/processed' if os.path.exists('../data') else 'data/processed'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'enriched_reviews.csv')
df.to_csv(output_file, index=False)

print(f"\n💾 Task 2 Pipeline Complete! Enriched file saved to: {output_file}")

In [ ]:
# 5. Display a Data Sample Matrix
print("\n📊 --- SENTIMENT DISTRIBUTION PER BANK ---")
display(pd.crosstab(df['bank'], df['sentiment']))

print("\n📋 --- SAMPLE DATA SNAPSHOT ---")
display(df[['bank', 'review', 'sentiment', 'theme']].head(3))